# 05 --- Task Decomposition

**CCA Pattern**: The coordinator decomposes queries into parallel and sequential phases based on data dependencies.

Web search and document analysis can run in parallel. Fact checking depends on their results, so it runs after.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.models.research import SubTask
from research_agents.agent.coordinator import sort_tasks_into_waves

## How `sort_tasks_into_waves()` Works

The topological sort in `coordinator.py` is straightforward:

```python
def sort_tasks_into_waves(tasks: list[SubTask]) -> list[list[SubTask]]:
    completed: set[str] = set()
    remaining = list(tasks)
    waves: list[list[SubTask]] = []
    while remaining:
        wave = [t for t in remaining
                if all(d in completed for d in t.depends_on)]
        # ... add wave, mark completed, continue
    return waves
```

Each iteration finds all tasks whose dependencies are already completed. Those tasks form the next wave and can run in parallel. If no tasks can run (circular dependency), all remaining tasks are placed in one wave as a fallback.

### The `depends_on` Field

The `SubTask.depends_on` field is a list of `task_id` strings. This is the **only** mechanism that creates sequential ordering. Tasks with empty `depends_on` are independent and can run in parallel.

## The Restaurant Kitchen Analogy

Task decomposition on an exam looks abstract, but the kitchen version makes it obvious.

A brigade kitchen has stations: appetizer, grill, pantry, pastry, and expediter. When an order comes in for a steak frites with a caesar salad:

- The **grill** station cooks the steak and the fries can run in parallel at the fry station -- neither needs the other's output.
- The **pantry** station builds the caesar salad in parallel with both of those -- independent inputs, independent tools.
- The **expediter** plates everything -- and that step must wait for *all* stations to finish. The plating `depends_on` every upstream step.

This is exactly what `sort_tasks_into_waves()` does. Independent stations (no `depends_on`) form Wave 0 and work simultaneously. The expediter, who needs everything, sits in Wave 1.

Now picture a kitchen where *nothing* is parallelized -- the same cook has to finish the steak before the fries can start, and the fries before the salad. Every dish comes out cold. That is the sequential-only anti-pattern.

## Anti-Pattern: Everything Sequential

Forcing every task to `depends_on` its predecessor -- even when no data actually flows between them -- is the most common decomposition mistake on the exam. It runs correctly but wastes the coordinator's biggest available optimization. Here's what it looks like:

In [ ]:
# ANTI-PATTERN: naive all-sequential decomposition
# Web search does not need database stats to start.
# Database stats do not need document analysis to start.
# But the author chained them anyway.
naive_tasks = [
    SubTask(task_id='web', agent_type='web_researcher',
        instruction='Search remote work studies', context=''),
    SubTask(task_id='data', agent_type='data_extractor',
        instruction='Query stats', context='',
        depends_on=['web']),                  # not actually needed!
    SubTask(task_id='docs', agent_type='document_analyzer',
        instruction='Parse Stanford study', context='',
        depends_on=['data']),                 # not actually needed!
    SubTask(task_id='facts', agent_type='fact_checker',
        instruction='Verify claims', context='',
        depends_on=['web', 'docs']),          # legitimately dependent
]
naive_waves = sort_tasks_into_waves(naive_tasks)
print(f'Anti-pattern wave count: {len(naive_waves)} (all single-task waves -- no parallelism)')
for i, wave in enumerate(naive_waves):
    print(f'  Wave {i}: {[t.task_id for t in wave]}')

### Why This Fails

Nothing is *wrong* here -- the pipeline still produces the right result. It just takes 4x as long as it needs to. On the exam, any option that chains `depends_on` without an actual data dependency is the distractor. The CCA answer always asks: *does this task actually need the previous task's output?* If no, drop the `depends_on` edge.

Below is the same research broken into the correct parallel + sequential shape. Compare the wave counts.

## Correct Pattern: Parallel + Sequential Waves

In [ ]:
# Define tasks with data dependencies
tasks = [
    SubTask(task_id='web', agent_type='web_researcher',
        instruction='Search for remote work studies',
        context='Focus on 2024'),
    SubTask(task_id='data', agent_type='data_extractor',
        instruction='Query remote work statistics',
        context='remote_work_stats table'),
    SubTask(task_id='docs', agent_type='document_analyzer',
        instruction='Parse Stanford study',
        context='doc-remote-work-stanford'),
    SubTask(task_id='facts', agent_type='fact_checker',
        instruction='Verify claims',
        context='', depends_on=['web', 'data', 'docs']),
]

waves = sort_tasks_into_waves(tasks)
for i, wave in enumerate(waves):
    ids = [t.task_id for t in wave]
    agents = [t.agent_type for t in wave]
    print(f'Wave {i}: {ids}')
    print(f'  Agents: {agents}')
    print(f'  Can run in parallel: {len(wave) > 1}')
    print()

### How the Coordinator Executes Waves

In `run_coordinator()`, each wave is processed sequentially, but tasks within a wave could be parallelized:

```python
for wave in waves:
    for task in wave:  # Could be concurrent
        context_string = build_subagent_context(task, results)
        result = run_agent_loop(client, services, context_string, ...)
        results[task.task_id] = result
```

The current implementation processes tasks within a wave sequentially for simplicity, but the wave structure makes it trivial to add concurrency later. The key point for the CCA exam is that the **dependency analysis is correct** -- tasks that can run in parallel are identified.

In [ ]:
# Compare: all sequential (no parallelism)
sequential_tasks = [
    SubTask(task_id='web', agent_type='web_researcher',
        instruction='a', context=''),
    SubTask(task_id='data', agent_type='data_extractor',
        instruction='b', context='', depends_on=['web']),
    SubTask(task_id='docs', agent_type='document_analyzer',
        instruction='c', context='', depends_on=['data']),
    SubTask(task_id='facts', agent_type='fact_checker',
        instruction='d', context='', depends_on=['docs']),
]
seq_waves = sort_tasks_into_waves(sequential_tasks)
print(f'Parallel waves: {len(waves)} (wave 0 has {len(waves[0])} tasks)')
print(f'Sequential waves: {len(seq_waves)} (all single-task waves)')

## CCA Exam Tip

> Task decomposition questions ask whether two subtasks should run in parallel or sequentially.
> - If Subtask B needs output from Subtask A -> sequential (`depends_on`)
> - If they work from independent inputs -> parallel (no `depends_on`)
> - 'Web search' and 'document parsing' are the canonical parallel pair
> - 'Fact checking' and 'report writing' are always sequential
> - The `SubTask.depends_on` field is the mechanism that creates ordering